In [1]:
import subprocess,sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "langchain", "langchain-groq", "langgraph", "langchain-core",
    "tavily-python", "beautifulsoup4", "requests",
    "fastapi", "uvicorn", "pyngrok", "nest-asyncio", "python-dotenv"],check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'langchain', 'langchain-groq', 'langgraph', 'langchain-core', 'tavily-python', 'beautifulsoup4', 'requests', 'fastapi', 'uvicorn', 'pyngrok', 'nest-asyncio', 'python-dotenv'], returncode=0)

In [2]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
os.environ["NGROK_TOKEN"] = userdata.get("NGROK_TOKEN")

In [3]:
import requests as http_requests
from bs4 import BeautifulSoup
from tavily import TavilyClient
from langchain.tools import tool
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.prebuilt import create_react_agent
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn, nest_asyncio, threading
from pyngrok import ngrok

In [4]:
from langchain.agents import create_agent

In [5]:
tavily_client = TavilyClient(api_key=os.environ['TAVILY_API_KEY'])

creating tools

In [6]:
@tool
def web_search(query: str) -> str:
  """Search the web for recent and reliable information. Returns titles, URLs and snippets."""
  results = tavily_client.search(query=query,max_results=3)
  out = []
  for r in results['results']:
    out.append(f"Title: {r['title']}\nURL: {r['url']}\nSnippet: {r['content'][:300]}\n")
  return "\n----\n".join(out)

In [7]:
@tool
def scrap_url(url: str) -> str:
  """Scrape and return clean text content from a given URL for deeper reading."""
  try:
        resp = http_requests.get(url, timeout=8, headers={"User-Agent": "Mozilla/5.0"})
        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()
        return soup.get_text(separator=" ", strip=True)[:1500]
  except Exception as e:
        return f"Could not scrape URL: {str(e)}"

creating agents

In [8]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=400
)

In [9]:
def build_search_agent():
  return create_agent(model=llm, tools=[web_search])

In [10]:
def build_reader_agent():
  return create_agent(model=llm, tools=[scrap_url])

propmts

In [11]:
writer_prompts = ChatPromptTemplate.from_messages([
     ("system", "You are an expert research writer. Write clear, structured and insightful reports."),
     ("human", """Write a detailed research report on the topic below.
      Topic: {topic}
      Research Gathered:
      {research}
      Structure the report as:
       - Introduction
       - Key Findings (minimum 3 well-explained points)
       - Conclusion
       - Sources (list all URLs found in the research)
      Be detailed, factual and professional.""")
])

creating writer chain

In [12]:
writer_chain = writer_prompts | llm | StrOutputParser()

In [13]:
critic_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a sharp and constructive research critic. Be honest and specific."),
    ("human", """Review the research report below and evaluate it strictly.
      Report:
      {report}
      Respond in this exact format:
      Score: X/10
      Strengths:
      - ...
      - ...
      Areas to Improve:
      - ...
      - ...
      One line verdict:
       ...""")
])

creating writer chain

In [14]:
critic_chain = critic_prompt | llm | StrOutputParser()

creating pipeline

In [15]:
def run_research_pipeline(topic: str) -> dict:
  state = {}
  search_agent = build_search_agent()
  result = search_agent.invoke({
      "messages":[("user", f"Find recent, reliable and detailed information about: {topic}")]
  })
  state["search_results"] = result["messages"][-1].content
  print("Search done\n", state["search_results"][:400], "...")

  reader_agent = build_reader_agent()
  result = reader_agent.invoke({
      "messages": [("user",
            f"Based on the following search results about '{topic}', "
            f"pick the most relevant URL and scrape it for deeper content.\n\n"
            f"Search Results:\n{state['search_results'][:800]}"
        )]
  })
  state["scraped_content"] = result["messages"][-1].content
  print("Search done\n", state["scraped_content"][:400], "...")

  research_combined = (
    f"SEARCH RESULTS:\n{state['search_results'][:1000]}\n\n"
    f"DETAILED SCRAPED CONTENT:\n{state['scraped_content'][:1500]}"
)
  state["report"] = writer_chain.invoke({
      "topic":topic,
      "research":research_combined
  })
  print("Report drafted",state['report'][:400],'...')

  state['feedback'] = critic_chain.invoke({
      "report": state["report"]
  })
  print("Critique done\n", state["feedback"])
  return state

FASTAPI CODE

In [16]:
app = FastAPI(title="AI Research Agent API")

# Allow requests from Streamlit Cloud
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class TopicRequest(BaseModel):
    topic: str

@app.get("/")
def health():
    return {"status": "ok", "message": "AI Research Agent API is running ✅"}

@app.post("/research")
def research(req: TopicRequest):
    try:
        result = run_research_pipeline(req.topic)
        return {
            "search_results":  result["search_results"],
            "scraped_content": result["scraped_content"],
            "report":          result["report"],
            "feedback":        result["feedback"],
        }
    except Exception as e:
        return {"error": str(e)}

print("FastAPI app ready")

# ── 8. Start Server + ngrok tunnel ───────────────────────────
nest_asyncio.apply()

ngrok.set_auth_token(os.environ["NGROK_TOKEN"])

# Kill any existing tunnels first
ngrok.kill()

public_url = ngrok.connect(8000)
public_url_str = str(public_url)

# ngrok returns NgrokTunnel object — extract the URL string
if hasattr(public_url, 'public_url'):
    public_url_str = public_url.public_url

print("\n" + "="*60)
print("🚀 SERVER IS LIVE!")
print("="*60)
print(f"📡 Public URL  :  {public_url_str}")
print(f"🔗 API Endpoint:  {public_url_str}/research")
print(f"❤️  Health Check:  {public_url_str}/")
print("="*60)
print("👆 Copy the API Endpoint URL and paste it into your Streamlit sidebar")
print("="*60 + "\n")

# Start uvicorn in a background thread so Colab cell doesn't block
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("✅ Server running in background — this cell is now done ✅")

FastAPI app ready

🚀 SERVER IS LIVE!
📡 Public URL  :  https://friskily-spiriferous-irene.ngrok-free.dev
🔗 API Endpoint:  https://friskily-spiriferous-irene.ngrok-free.dev/research
❤️  Health Check:  https://friskily-spiriferous-irene.ngrok-free.dev/
👆 Copy the API Endpoint URL and paste it into your Streamlit sidebar

✅ Server running in background — this cell is now done ✅
